# Evaluating the Segmentation Models

In [1]:
# ======================
# Load Libraries
# ======================

import os
import re
import torch
import numpy as np
import pandas as pd
from skimage.io import imread
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm import tqdm
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

import sys
sys.path.append('../')
from utils.models.uNet import UNet
from utils.models.SegFormer import segformer
from utils.models.deeplabv3p import DeepLabV3Plus
from utils.models.SegNet import segnet
from utils.models.maskFormer import MaskFormer
from utils.models.resnet101 import resnet101_backbone

c:\Users\gnoceras\Documents\SegmentationStudy\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\gnoceras\Documents\SegmentationStudy\.venv\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Common things

In [2]:
class SegEvalDataset(Dataset):
    def __init__(self, image_dir, mask_dir, image_files):
        self.image_dir = image_dir
        self.mask_dir = mask_dir

        # keep only files with a non-empty mask
        valid_files = []
        for img_name in image_files:
            mask_name = os.path.splitext(img_name)[0] + ".png"
            mask_path = os.path.join(self.mask_dir, mask_name)
            if not os.path.exists(mask_path):
                # skip if mask missing
                continue
            try:
                mask = imread(mask_path)
            except Exception:
                # skip unreadable masks
                continue
            if mask.ndim == 3:
                mask = mask[..., 0]
            # treat any non-zero pixel as non-empty
            if np.sum(mask) > 0:
                valid_files.append(img_name)
        self.image_files = valid_files

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        mask_name = os.path.splitext(img_name)[0] + ".png"
        image = imread(os.path.join(self.image_dir, img_name))
        mask = imread(os.path.join(self.mask_dir, mask_name))
        if mask.ndim == 3:
            mask = mask[..., 0]
        image = TF.to_tensor(image)
        image = TF.normalize(image, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        mask = (mask > 0).astype(np.float32)
        mask = torch.from_numpy(mask).float().unsqueeze(0)
        return image, mask, img_name

In [3]:
# ======================
# Utility functions
# ======================

def iou_score(y_true, y_pred):
    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - intersection
    return intersection / (union + 1e-7)

def compute_metrics(y_true, y_pred):
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    
    precision = precision_score(y_true_flat, y_pred_flat, zero_division=0)
    recall = recall_score(y_true_flat, y_pred_flat, zero_division=0)
    f1 = f1_score(y_true_flat, y_pred_flat, zero_division=0)
    iou = iou_score(y_true, y_pred)

    return iou, precision, recall, f1

In [4]:
# ======================
# Model loading stubs
# ======================

def load_model(crop, model_name, device):
    if model_name == "U-NET":
        channels = [32, 64, 128, 256, 512]
        model = UNet(in_channels=3, out_channels=1, channels=channels, bilinear=True, use_batchnorm=True).to(device)
    elif model_name == "SegFormer":
        model = segformer(in_channels = 3, num_classes = 1).to(device)
    elif model_name == "DeepLabV3Plus":
        model = DeepLabV3Plus(num_classes=1, output_stride=16, backbone_width_mult=1.0).to(device)
    elif model_name == "SegNet":
        model = segnet(in_channels=3, num_classes=1, pretrained=False).to(device)
    elif model_name == "MaskFormer":
        resnet101 = resnet101_backbone()
        model = MaskFormer(
            backbone=resnet101, 
                num_classes=1, 
                num_queries=5,
                embed_dim=64, 
                transformer_layers=1,
                transformer_heads=2,
                transformer_ffn_dim=256,
                return_binary=True
            ).to(device)
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    model.load_state_dict(torch.load(f'../models/{crop}_{model_name}_seg.pt', map_location=device))
    model.eval()
    return model

def predict_mask(model, image, device):
    img_input = image.unsqueeze(0).to(device)
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            pred_logits = model(img_input)
            pred_mask = (torch.sigmoid(pred_logits) > 0.5).float().cpu().squeeze().numpy()
    return pred_mask

## Using initial trained models

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "../data/"
crops = ["sorghum", "wheat", "corn"]
models = ["DeepLabV3Plus", "SegFormer", "SegNet", "U-NET", "MaskFormer"]

In [8]:
# ======================
# Main evaluation
# ======================

results = []

BATCH_SIZE = 8

for crop in crops:
    image_dir = os.path.join(data_dir, crop, "test", "images")
    mask_dir = os.path.join(data_dir, crop, "test", "masks")
    image_files = [f for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

    dataset = SegEvalDataset(image_dir, mask_dir, image_files)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    for model_name in models:
        print(f"Evaluating {model_name} on {crop}...")
        model = load_model(crop, model_name, device)
        model.eval()

        for images, masks, img_names in tqdm(loader):
            images = images.to(device)
            masks_np = masks.squeeze(1).cpu().numpy()  # [B, H, W]
            with torch.no_grad():
                with torch.amp.autocast('cuda'):
                    logits = model(images)
                    preds = (torch.sigmoid(logits) > 0.5).float().cpu().numpy().squeeze(1)  # [B, H, W]

            for i in range(images.size(0)):
                mask = masks_np[i]
                pred = preds[i]
                img_name = img_names[i]
                match = re.match(r'(\d+)_([^_]+)_\d+(_aug)?', img_name)
                if match:
                    collectiondate, genotype, aug = match.groups()
                    augmented = "yes" if aug else "no"
                else:
                    collectiondate, genotype, augmented = "unknown", "unknown", "unknown"
                iou, precision, recall, f1 = compute_metrics(mask, pred)
                results.append({
                    "crop": crop,
                    "collectiondate": collectiondate,
                    "genotype": genotype,
                    "augmented?": augmented,
                    "modelname": model_name,
                    "IoU": iou,
                    "Precision": precision,
                    "Recall": recall,
                    "F1": f1,
                })
        torch.cuda.empty_cache()

Evaluating DeepLabV3Plus on sorghum...


100%|██████████| 861/861 [12:49<00:00,  1.12it/s]


Evaluating SegFormer on sorghum...


100%|██████████| 861/861 [13:15<00:00,  1.08it/s]


Evaluating SegNet on sorghum...


...

100%|██████████| 663/663 [10:00<00:00,  1.10it/s]


Evaluating U-NET on corn...


100%|██████████| 663/663 [09:37<00:00,  1.15it/s]


Evaluating MaskFormer on corn...


c:\Users\gnoceras\Documents\SegmentationStudy\.venv\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
100%|██████████| 663/663 [09:16<00:00,  1.19it/s]


In [9]:
# ======================
# Save to CSV
# ======================

df = pd.DataFrame(results)
df.to_csv("../results/model_evaluation_results.csv", index=False)
print("✅ Evaluation completed and saved to results/model_evaluation_results.csv")

✅ Evaluation completed and saved to results/model_evaluation_results.csv


## Trained in one collection date and testing in others

In [2]:
def load_model(crop, model_name, device, train_date):
    if model_name == "U-NET":
        from utils.models.uNet import UNet
        channels = [32, 64, 128, 256, 512]
        model = UNet(in_channels=3, out_channels=1, channels=channels, bilinear=True, use_batchnorm=True).to(device)
    elif model_name == "SegFormer":
        from utils.models.SegFormer import segformer
        model = segformer(in_channels=3, num_classes=1).to(device)
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    model_path = f"../models/{crop}_{model_name}_date{train_date}_seg.pt"
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "../data/"
crop = "sorghum"  # "wheat" "sorghum" or "corn"
model_name = "SegFormer"  # or "U-NET"
collection_dates = ["1", "2", "3", "4", "5"]
BATCH_SIZE = 8

results = []

In [6]:
for train_date in collection_dates:
    # Load model trained on train_date
    model = load_model(crop, model_name, device, train_date)
    print(f"Model loaded for training date {train_date}")

    # Infer on all dates, including the training date
    for test_date in collection_dates:
        image_dir = os.path.join(data_dir, crop, "test", "images")
        mask_dir = os.path.join(data_dir, crop, "test", "masks")
        # Filter images by test_date
        image_files = [f for f in os.listdir(image_dir) if f.startswith(f"{test_date}_") and f.endswith(('.png', '.jpg', '.jpeg'))]
        if not image_files:
            print(f"No images found for test date {test_date}")
            continue
        dataset = SegEvalDataset(image_dir, mask_dir, image_files)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

        for images, masks, img_names in tqdm(loader, desc=f"Infer {train_date}→{test_date}"):
            images = images.to(device)
            masks_np = masks.squeeze(1).cpu().numpy()
            with torch.no_grad():
                with torch.amp.autocast('cuda'):
                    logits = model(images)
                    preds = (torch.sigmoid(logits) > 0.5).float().cpu().numpy().squeeze(1)
            for i in range(images.size(0)):
                mask = masks_np[i]
                pred = preds[i]
                img_name = img_names[i]
                match = re.match(r'(\d+)_([^_]+)_(\d+)(_aug)?', img_name)
                if match:
                    collectiondate, genotype, individual_id, aug = match.groups()
                else:
                    collectiondate, genotype, individual_id = "unknown", "unknown", "unknown"
                iou = iou_score(mask, pred)
                results.append({
                    "crop": crop,
                    "modelname": model_name,
                    "train_date": train_date,
                    "test_date": test_date,
                    "collectiondate": collectiondate,
                    "genotype": genotype,
                    "individual_id": individual_id,
                    "img_name": img_name,
                    "IoU": iou
                })

Model loaded for training date 1


Infer 1→5: 100%|██████████| 178/178 [00:54<00:00,  3.28it/s]


Model loaded for training date 2


Infer 2→5: 100%|██████████| 178/178 [01:15<00:00,  2.35it/s]


Model loaded for training date 3


Infer 3→5: 100%|██████████| 178/178 [00:51<00:00,  3.44it/s]


Model loaded for training date 4


Infer 4→5: 100%|██████████| 178/178 [00:54<00:00,  3.29it/s]


Model loaded for training date 5


Infer 5→5: 100%|██████████| 178/178 [00:48<00:00,  3.71it/s]


In [7]:
df = pd.DataFrame(results)
df.to_csv("../results/model_evaluation_crossDate_sorghum.csv", index=False)
print("✅ Cross-date IoU results saved to results/model_evaluation_crossDate_sorghum.csv")

✅ Cross-date IoU results saved to results/model_evaluation_crossDate_sorghum.csv


## Cross Genotypes

In [5]:
def load_model(crop, model_name, device, genotype):
    if model_name == "U-NET":
        from utils.models.uNet import UNet
        channels = [32, 64, 128, 256, 512]
        model = UNet(in_channels=3, out_channels=1, channels=channels, bilinear=True, use_batchnorm=True).to(device)
    elif model_name == "SegFormer":
        from utils.models.SegFormer import segformer
        model = segformer(in_channels=3, num_classes=1).to(device)
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    model_path = f"../models/{crop}_{model_name}_{genotype}_seg.pt"
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "../data/"
crop = "sorghum"  # or "wheat" or "sorghum" or "corn"
model_name = "SegFormer"  # or "U-NET"
test_genotypes = list(range(1, 81))
train_genotype = "second_half"
BATCH_SIZE = 8

In [7]:
results = []

In [18]:
model = load_model(crop, model_name, device, train_genotype)
print(f"Model loaded for genotype {train_genotype}")

# Infer on all other genotypes
for test_genotype in test_genotypes:
    image_dir = os.path.join(data_dir, crop, "test", "images")
    mask_dir = os.path.join(data_dir, crop, "test", "masks")
    # Filter images by test_genotype
    image_files = [f for f in os.listdir(image_dir) if f.split('_')[1] == str(test_genotype) and f.endswith(('.png', '.jpg', '.jpeg'))]
    if not image_files:
        print(f"No images found for test genotype {test_genotype}")
        continue
    dataset = SegEvalDataset(image_dir, mask_dir, image_files)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    for images, masks, img_names in tqdm(loader, desc=f"Infer {train_genotype}→{test_genotype}"):
        images = images.to(device)
        masks_np = masks.squeeze(1).cpu().numpy()
        with torch.no_grad():
            with torch.amp.autocast('cuda'):
                logits = model(images)
                preds = (torch.sigmoid(logits) > 0.5).float().cpu().numpy().squeeze(1)
        for i in range(images.size(0)):
            mask = masks_np[i]
            pred = preds[i]
            img_name = img_names[i]
            # Parse genotype and id if needed
            match = re.match(r'(\d+)_([^_]+)_(\d+)(_aug)?', img_name)
            if match:
                collectiondate, genotype, individual_id, aug = match.groups()
            else:
                collectiondate, genotype, individual_id = "unknown", "unknown", "unknown"
            iou, precision, recall, f1 = compute_metrics(mask, pred)
            results.append({
                "crop": crop,
                "modelname": model_name,
                "train_genotype": train_genotype,
                "test_genotype": test_genotype,
                "collectiondate": collectiondate,
                "genotype": genotype,
                "individual_id": individual_id,
                "img_name": img_name,
                "IoU": iou,
                "Precision": precision,
                "Recall": recall,
                "F1": f1,
            })

Model loaded for genotype second_half


Infer second_half→7: 100%|██████████| 3/3 [00:02<00:00,  1.46it/s]


No images found for test genotype 8


Infer second_half→80: 100%|██████████| 11/11 [00:08<00:00,  1.27it/s]


In [20]:
df = pd.DataFrame(results)
df.to_csv("../results/model_evaluation_crossGenotype.csv", index=False)
print("✅ Cross-genotype IoU results saved to results/model_evaluation_crossGenotype.csv")

✅ Cross-genotype IoU results saved to results/model_evaluation_crossGenotype.csv


## Transfer Learning

In [5]:
def load_model(crop, model_name, device):
    if model_name == "U-NET":
        from utils.models.uNet import UNet
        channels = [32, 64, 128, 256, 512]
        model = UNet(in_channels=3, out_channels=1, channels=channels, bilinear=True, use_batchnorm=True).to(device)
    elif model_name == "SegFormer":
        from utils.models.SegFormer import segformer
        model = segformer(in_channels=3, num_classes=1).to(device)
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    model_path = f"../models/{crop}_{model_name}_transfer_seg.pt"
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "../data/"
crops = ["sorghum"]  # or "wheat"
models = ["SegFormer"]  # or "U-NET"
BATCH_SIZE = 8

#results = []

In [9]:
# ======================
# Main evaluation
# ======================

for crop in crops:
    image_dir = os.path.join(data_dir, crop, "test", "images")
    mask_dir = os.path.join(data_dir, crop, "test", "masks")
    image_files = [f for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

    dataset = SegEvalDataset(image_dir, mask_dir, image_files)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    for model_name in models:
        print(f"Evaluating {model_name} on {crop}...")
        model = load_model(crop, model_name, device)
        model.eval()

        for images, masks, img_names in tqdm(loader):
            images = images.to(device)
            masks_np = masks.squeeze(1).cpu().numpy()  # [B, H, W]
            with torch.no_grad():
                with torch.amp.autocast('cuda'):
                    logits = model(images)
                    preds = (torch.sigmoid(logits) > 0.5).float().cpu().numpy().squeeze(1)  # [B, H, W]

            for i in range(images.size(0)):
                mask = masks_np[i]
                pred = preds[i]
                img_name = img_names[i]
                match = re.match(r'(\d+)_([^_]+)_\d+(_aug)?', img_name)
                if match:
                    collectiondate, genotype, aug = match.groups()
                    augmented = "yes" if aug else "no"
                else:
                    collectiondate, genotype, augmented = "unknown", "unknown", "unknown"
                iou = iou_score(mask, pred)
                results.append({
                    "crop": crop,
                    "collectiondate": collectiondate,
                    "genotype": genotype,
                    "augmented?": augmented,
                    "modelname": model_name,
                    "IoU": iou,
                })

Evaluating SegFormer on sorghum...


100%|██████████| 861/861 [03:09<00:00,  4.55it/s]


In [10]:
df = pd.DataFrame(results)
df.to_csv("../results/model_evaluation_transferLearning.csv", index=False)
print("✅ Cross-genotype IoU results saved to results/model_evaluation_transferLearning.csv")

✅ Cross-genotype IoU results saved to results/model_evaluation_transferLearning.csv
